In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualização gráfica inline no Jupyter notebook
%matplotlib inline

# Comparar pipelines em partições pareadas reais de sujeitos

Compare a regressão logística e o LDA com encolhimento (*shrinkage*) usando participantes
retidos (*held-out*) idênticos. Pareie as pontuações por participante; ensaios não são
réplicas independentes para uma comparação populacional.

Dados: Nakanishi2015, NEMAR ``nm000118``, sujeitos 1–3, sessão 0,
execução 0: aproximadamente 21.1 MB no primeiro download. Defina ``EEGDASH_CACHE_DIR``
para reutilizar o cache. Esta versão processada já inclui filtragem,
redução da taxa de amostragem (*downsampling*) e tratamento de latência; não adicione outra correção de latência.
Consulte o [estudo de origem](https://doi.org/10.1371/journal.pone.0140703)
e a [versão NEMAR](https://nemar.org/dataset/nm000118).

Pré-requisitos: o loop de validação LOSO do tutorial 51 e pipelines do scikit-learn. Esta página
carrega suas próprias gravações e requer o SciPy para o teste pareado opcional.
A pergunta central é se um classificador fixo melhora as pontuações dos mesmos participantes.
Nenhum dos classificadores é selecionado ou ajustado usando esses resultados de teste.


## 1. Selecionar uma coorte pequena e explícita
Filtrar sujeitos, sessão e execução delimita o download. Cortar (*crop*) após
abrir uma gravação reduziria a computação, mas não o tamanho do download.



In [ ]:
# Importa módulos de sistema operacional, funções parciais e caminhos no sistema de arquivos
import os
from functools import partial
from pathlib import Path

# Importa bibliotecas para plotagem, manipulação de arrays numéricos e DataFrames
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa gerador de janelas baseadas em eventos da Braindecode
from braindecode.preprocessing import create_windows_from_events
# Importa modelo de regressão logística e métricas do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
# Importa divisão Leave-One-Group-Out, pipeline e padronizador de escala
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset e extratores espectrais do EEGDash
from eegdash import EEGDashDataset
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_bands_power,
    spectral_preprocessor,
)

# Define o caminho do diretório de cache
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Lista dos três sujeitos para o teste pareado
subjects = ["1", "2", "3"]
# Inicializa e carrega os dados brutos de SSVEP dos sujeitos
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="nm000118",
    subject=subjects,
    session="0",
    run="0",
    task="ssvep",
    n_jobs=1,
)
# Assegura a presença de uma gravação por participante
assert len(dataset.datasets) == len(subjects), "Expected one recording per subject"
# Exibe os metadados descritivos das gravações carregadas
print(dataset.description[["subject", "session", "run"]])

## 2. Inspecionar anotações reais e verificar o contrato do sinal
Acessar ``raw`` baixa aquela gravação. Nomes de anotação identificam a
frequência do estímulo atendido em Hz; eles fornecem cada rótulo de classificação.
Todos os participantes devem ter a mesma ordem de canais e frequência de amostragem.



In [ ]:
# Acessa os dados da primeira gravação para checar parâmetros de amostragem e canais
raw = dataset.datasets[0].raw
sfreq = raw.info["sfreq"]
channel_names = raw.ch_names
# Identifica as 12 frequências a partir das anotações ordenadas numericamente
class_names = sorted(set(raw.annotations.description), key=float)
# Mapeia nomes das classes para índices numéricos de 0 a 11
mapping = {name: index for index, name in enumerate(class_names)}
# Garante que todas as doze frequências de estímulo SSVEP estão presentes
assert len(mapping) == 12, "Expected the twelve SSVEP stimulus frequencies"
# Valida consistência de canais, frequência de amostragem e anotações em cada sujeito
for recording in dataset.datasets:
    recording_raw = recording.raw
    assert recording_raw.ch_names == channel_names
    assert recording_raw.info["sfreq"] == sfreq
    assert set(recording_raw.annotations.description) == set(mapping)
# Imprime informações de canais, taxa de amostragem e frequências mapeadas
print(f"Channels: {channel_names}; sampling frequency: {sfreq} Hz")
print("Stimulus frequencies (Hz):", class_names)

## 3. Fazer uma janela de quatro segundos por ensaio anotado
Cada intervalo anotado dura 4.15 segundos. Mantenha seus primeiros quatro segundos e
descarte o restante. Tamanho e passo (*stride*) explícitos evitam janelas sobrepostas
ou a extensão da época além da duração do evento gravado.
A 256 Hz, quatro segundos contêm 1.024 amostras. O array resultante possui
eixos (540 ensaios, 8 canais de EEG, 1.024 amostras), com dados expressos em volts.
Os metadados possuem uma linha por linha do array. ``target`` é um índice de classe, não uma
frequência em Hz; ``mapping`` é a conversão explícita entre eles.
A fonte contém 15 ensaios de cada uma das 12 frequências por pessoa.
Uma classe ausente é uma quebra no contrato de dados, não um motivo para re-rotular ensaios.



In [ ]:
# Define tamanho da janela correspondente a 4 segundos em amostras (1024 amostras a 256 Hz)
window_size = int(4 * sfreq)
# Cria as janelas a partir dos eventos descartando a fração residual final com on_last_window='drop'
windows = create_windows_from_events(
    dataset,
    mapping=mapping,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=window_size,
    window_stride_samples=window_size,
    on_last_window="drop",
    preload=True,
)
# Extrai tabela de metadados das janelas
metadata = windows.get_metadata()
# Confirma que cada ensaio gerou exatamente uma janela
assert (metadata.i_window_in_trial == 0).all(), "Expected one window per trial"
# Assegura ausência de ensaios duplicados
assert not metadata.duplicated(["subject", "session", "run", "i_start_in_trial"]).any()
# Extrai vetor numérico de classes alvo
y = metadata["target"].to_numpy(dtype=int)
# Extrai vetor com identificadores dos sujeitos
groups = metadata["subject"].astype(str).to_numpy()
# Empilha os dados em uma matriz 3D (540 ensaios, 8 canais, 1024 amostras)
X = np.stack([window[0] for window in windows])
# Valida dimensões, sujeitos presentes e finitude numérica de X
assert X.shape == (len(metadata), len(channel_names), window_size)
assert set(groups) == set(subjects)
assert np.isfinite(X).all()
# Exibe tabela de contingência cruzando sujeito por classe de estímulo
print(pd.crosstab(groups, y, rownames=["subject"], colnames=["class"]))

## 4. Extrair características espectrais de cada janela
As respostas de SSVEP contêm energia na frequência do estímulo. Use o log da potência espectral
em torno de cada frequência de estímulo, mantendo todos os oito canais posteriores.
Esta transformação por janela não aprende nada de outros ensaios ou sujeitos.
O escalonador abaixo, por outro lado, deve ser ajustado apenas nos sujeitos de treino.
O pré-processador espectral compartilhado do EEGDash calcula uma PSD de Welch com um segmento Hann
de quatro segundos e bins de 0.25 Hz. Cada banda estreita é centrada
em uma frequência de estímulo documentada; esses centros definem a tarefa, não
preditores específicos de ensaios. Reter oito canais fornece 12 × 8 = 96
características. A versão já processada de SSVEP não precisa de outro passe de limpeza
do EEGPrep ou correção de latência visual.

``spectral_bands_power`` soma os bins selecionados da PSD. Multiplicar pelo espaçamento
de 0.25 Hz converte V²/Hz para a potência aproximada da banda em V². O log comprime
essa escala; o StandardScaler ainda é ajustado apenas nos participantes de treino.



In [ ]:
# Define bandas estreitas de +/- 0.125 Hz ao redor de cada frequência de estímulo anotada
bands = {
    f"hz_{name}": (float(name) - 0.125, float(name) + 0.125) for name in class_names
}
# Configura extrator com PSD Welch de 4 segundos e resolução de 0.25 Hz na faixa de 8 a 16 Hz
spectral = FeatureExtractor(
    {"power": partial(spectral_bands_power, bands=bands)},
    preprocessor=partial(
        spectral_preprocessor,
        fs=sfreq,
        nperseg=window_size,
        noverlap=0,
        f_min=8,
        f_max=16,
    ),
)
# Executa a extração em lote para todas as janelas
feature_table = extract_features(
    windows, {"spectral": spectral}, batch_size=64, n_jobs=1
).to_dataframe()
# Valida o número total de colunas de características (12 bandas * 8 canais = 96)
assert feature_table.shape == (len(y), len(class_names) * len(channel_names))
# Converte densidade para potência em V² multiplicando pelo bin width (0.25 Hz) e aplica transformação log
features = np.log(np.maximum(feature_table.to_numpy() * sfreq / window_size, 1e-30))
# Valida que todos os valores calculados são finitos
assert np.isfinite(features).all()

## 5. Avaliar ambos os classificadores exatamente nas mesmas partições LOSO
Ambos os pipelines ajustam seu escalonador apenas nos participantes de treino. Todas as escolhas
são fixadas antes da avaliação; calibre alternativas dentro das partições de treino.



In [ ]:
# Importa o teste estatístico não paramétrico de Wilcoxon do SciPy
from scipy.stats import wilcoxon
# Importa o classificador de Análise Discriminante Linear (LDA) do scikit-learn
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

O LDA com encolhimento (*shrinkage*) regulariza sua estimativa de covariância compartilhada entre classes, o que ajuda
quando os bins espectrais são correlacionados. ``solver="lsqr"`` suporta esse encolhimento;
``shrinkage="auto"`` estima sua força a partir da partição de treino. A regressão
logística utiliza sua penalidade L2 padrão. Ambos consomem as mesmas 96 características
e exatamente a mesma atribuição de 360 ensaios de treino e 180 de teste em cada partição.



In [ ]:
# Inicializa lista para registrar as métricas pareadas de cada partição
rows = []
# Executa validação cruzada deixando um participante de fora (LOSO)
for train, test in LeaveOneGroupOut().split(features, y, groups):
    # Garante isolamento estrito entre os participantes de treino e teste
    assert set(groups[train]).isdisjoint(groups[test])
    # Garante que todas as doze classes ocorrem tanto no treino quanto no teste
    assert set(y[train]) == set(y[test]) == set(mapping.values())
    # Inicializa o registro identificando o sujeito mantido fora para teste
    row = {"subject": groups[test][0]}
    # Itera comparando os dois classificadores sob idênticas condições
    for name, classifier in {
        "Logistic": LogisticRegression(max_iter=1000),
        "LDA": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
    }.items():
        # Constrói o pipeline contendo padronizador e o classificador da vez
        model = make_pipeline(StandardScaler(), classifier)
        # Ajusta o modelo estritamente nos dados de treino da partição atual
        model.fit(features[train], y[train])
        # Calcula a acurácia balanceada nos dados do sujeito de teste retido
        row[name] = balanced_accuracy_score(y[test], model.predict(features[test]))
    rows.append(row)
# Monta a tabela final indexada pelo identificador do sujeito
results = pd.DataFrame(rows).set_index("subject")
# Valida que todos os três sujeitos foram processados com índices únicos
assert len(results) == len(subjects) and results.index.is_unique
# Exibe os resultados comparativos
print(results)
# Calcula a diferença pareada das pontuações (LDA menos Regressão Logística)
difference = results["LDA"] - results["Logistic"]
print("Paired differences (LDA minus Logistic):", difference.to_dict())
# Com três participantes, um teste bicaudal tem resolução muito baixa. A inferência
# exata de postos sinalizados abaixo exige diferenças não nulas com postos distintos;
# relate apenas as diferenças medidas caso zeros ou empates violem esse caso.
if (difference != 0).all() and difference.abs().is_unique:
    print("Exploratory exact Wilcoxon:", wilcoxon(difference, method="exact"))
else:
    print("Zeros or tied absolute differences: report paired differences only.")

## 6. Conectar as duas pontuações para cada participante



In [ ]:
# Plota linhas conectando as pontuações de cada participante entre os dois modelos
for subject, row in results.iterrows():
    plt.plot(["Logistic", "LDA"], row, "o-", label=f"Subject {subject}")
# Configura rótulos, limites do eixo Y e legenda
plt.ylabel("LOSO balanced accuracy")
plt.ylim(0, 1)
plt.legend()
# Exibe o gráfico
plt.show()

## 7. Interpretar a comparação pareada
Uma linha conectora tem inclinação ascendente quando o LDA melhora a sensibilidade média
de classe daquele participante; uma linha descendente favorece a regressão logística. O pareamento elimina
a comparação enganosa que surgiria de testar os dois modelos em pessoas diferentes. A unidade
de replicação permanece sendo o participante, não cada um dos 540 ensaios.

O cálculo dos postos sinalizados usa as magnitudes e os sinais de três diferenças pareadas
e assume uma distribuição simétrica de diferenças para sua interpretação habitual de localização.
Diferenças absolutas não nulas e distintas permitem o cálculo exato para pequenas amostras usado aqui.
Três pares não podem sustentar evidência forte: mesmo com todas as diferenças na mesma direção,
o valor-p exato bicaudal é 0.25. Relate tamanhos de efeito e participantes independentes adicionais
antes de afirmar uma vantagem confiável.

